In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import subprocess, re
import opensmile

WIN_DIR = Path("window")
AUDIO_DIR = Path("audio")

OUT_DIR = Path("out/audio")
OUT_DIR.mkdir(parents=True, exist_ok=True)

WIN = 4.0
HOP = 1.0

EPS = 1e-9

print("WIN_DIR:", WIN_DIR)
print("AUDIO_DIR:", AUDIO_DIR)
print("OUT_DIR:", OUT_DIR)


In [ ]:
def make_smile_lld():
    return opensmile.Smile(
        feature_set=opensmile.FeatureSet.eGeMAPSv02,
        feature_level=opensmile.FeatureLevel.LowLevelDescriptors,
    )

def to_seconds_index(df: pd.DataFrame) -> pd.DataFrame:
    if not isinstance(df, pd.DataFrame) or len(df) == 0:
        return df

    idx = df.index

    if isinstance(idx, pd.TimedeltaIndex):
        out = df.copy()
        out.index = idx.total_seconds().astype(float)
        out.index.name = "t"
        return out

    if isinstance(idx, pd.MultiIndex):
        names = [n if n is not None else "" for n in idx.names]
        t = idx.get_level_values("start") if "start" in names else idx.get_level_values(-1)

        out = df.copy()
        out.index = (
            t.total_seconds().astype(float)
            if isinstance(t, pd.TimedeltaIndex)
            else t.astype(float)
        )
        out.index.name = "t"
        return out

    out = df.copy()
    out.index = out.index.astype(float)
    out.index.name = "t"
    return out

def get_wav_duration_seconds(wav_path: str) -> float:
    p = subprocess.run(
        ["ffprobe", "-hide_banner", "-i", wav_path],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
    )
    m = re.search(r"Duration:\s*(\d+):(\d+):(\d+\.\d+)", p.stderr)
    if not m:
        raise RuntimeError(f"Could not parse duration: {wav_path}")
    hh, mm, ss = m.groups()
    return int(hh) * 3600 + int(mm) * 60 + float(ss)

def lld_chunk_to_seconds_index(df: pd.DataFrame, chunk_start_s: float) -> pd.DataFrame:
    if df is None or len(df) == 0:
        return df
    df = to_seconds_index(df)
    out = df.copy()
    out.index = out.index + float(chunk_start_s)
    out.index.name = "t"
    return out

def extract_llds_in_chunks(
    smile,
    wav_path: str,
    total_duration_s: float,
    chunk_s: float = 60.0,
    overlap_s: float = 0.0,
) -> pd.DataFrame:
    dfs = []
    step = max(1e-6, chunk_s - overlap_s)
    starts = np.arange(0.0, total_duration_s, step)

    for s in starts:
        e = min(total_duration_s, s + chunk_s)
        df = smile.process_file(wav_path, start=float(s), end=float(e))
        if df is None or len(df) == 0:
            continue
        dfs.append(lld_chunk_to_seconds_index(df, s))

    if not dfs:
        return pd.DataFrame()

    lld = pd.concat(dfs, axis=0).sort_index()
    if overlap_s > 0:
        lld = lld[~lld.index.duplicated(keep="first")]
    return lld

def build_windows_for_video(t_max: float, win: float, hop: float) -> pd.DataFrame:
    starts = (
        np.array([0.0])
        if t_max <= win
        else np.arange(0.0, (t_max - win) + 1e-6, hop)
    )
    return pd.DataFrame({
        "w_start": starts,
        "w_end": starts + win,
        "w_center": starts + win / 2.0,
    })

def aggregate_llds_to_windows(
    lld: pd.DataFrame,
    windows: pd.DataFrame,
    stats=("mean", "std", "min", "max", "median"),
) -> pd.DataFrame:
    cols = list(lld.columns)
    rows = []

    for w in windows.itertuples(index=False):
        seg = lld[(lld.index >= w.w_start) & (lld.index < w.w_end)]
        row = {"w_start": w.w_start, "w_end": w.w_end, "w_center": w.w_center}

        if seg.empty:
            for c in cols:
                for s in stats:
                    row[f"{c}_{s}"] = np.nan
        else:
            arr = seg.to_numpy(dtype=float)
            for j, c in enumerate(cols):
                x = arr[:, j]
                if "mean" in stats:   row[f"{c}_mean"] = float(np.nanmean(x))
                if "std" in stats:    row[f"{c}_std"] = float(np.nanstd(x))
                if "min" in stats:    row[f"{c}_min"] = float(np.nanmin(x))
                if "max" in stats:    row[f"{c}_max"] = float(np.nanmax(x))
                if "median" in stats: row[f"{c}_median"] = float(np.nanmedian(x))

        rows.append(row)

    return pd.DataFrame(rows)

def add_basic_validity_flags(feat_df: pd.DataFrame) -> pd.DataFrame:
    feat_cols = [c for c in feat_df.columns if c not in {"w_start", "w_end", "w_center"}]
    out = feat_df.copy()
    out["has_any_audio_feature"] = (~out[feat_cols].isna()).any(axis=1).astype(int)
    return out


In [ ]:
w_train = pd.read_csv(WIN_DIR / "windows_train.csv")
w_val   = pd.read_csv(WIN_DIR / "windows_val.csv")
w_test  = pd.read_csv(WIN_DIR / "windows_test.csv")

all_vids = sorted(set(w_train.video_id) | set(w_val.video_id) | set(w_test.video_id))
print("Total videos to process:", len(all_vids))

smile = make_smile_lld()

all_feats = []
failed = []

for vid in all_vids:
    wav_path = AUDIO_DIR / f"{vid}.wav"
    if not wav_path.exists():
        failed.append((vid, "missing_wav"))
        continue

    try:
        dur = get_wav_duration_seconds(str(wav_path))
        lld = extract_llds_in_chunks(smile, str(wav_path), dur, chunk_s=60.0, overlap_s=0.0)

        if lld is None or len(lld) == 0:
            failed.append((vid, "lld_empty_after_chunking"))
            continue

        t_max = float(lld.index.max())
        windows = build_windows_for_video(t_max, WIN, HOP)

        feat_df = aggregate_llds_to_windows(lld, windows)
        feat_df = add_basic_validity_flags(feat_df)

        feat_df.insert(0, "video_id", vid)
        all_feats.append(feat_df)

    except Exception as e:
        failed.append((vid, repr(e)))
        print(f"❌ {vid}: {e}")


audio_windows = pd.concat(all_feats, ignore_index=True) if all_feats else pd.DataFrame()
audio_windows.to_csv(OUT_DIR / "audio_windows_features.csv", index=False)

print("Saved:", OUT_DIR / "audio_windows_features.csv")
print("Failed:", len(failed))
failed[:10]


In [ ]:
audio_windows = pd.read_csv(OUT_DIR / "audio_windows_features.csv")

# merge helper
def merge_audio(w_df: pd.DataFrame, audio_df: pd.DataFrame) -> pd.DataFrame:
    merged = w_df.merge(
        audio_df,
        on=["video_id", "w_start", "w_end"],
        how="left",
        validate="many_to_one"
    )
    return merged

train_merged = merge_audio(w_train, audio_windows)
val_merged   = merge_audio(w_val, audio_windows)
test_merged  = merge_audio(w_test, audio_windows)

def report_merge(name, df):
    feat_cols = [c for c in df.columns if c.endswith("_mean")]  # rough proxy
    miss = df[feat_cols].isna().all(axis=1).mean() if feat_cols else np.nan
    print(f"{name}: rows={len(df)}  windows_missing_all_features={miss:.3f}")

report_merge("TRAIN", train_merged)
report_merge("VAL",   val_merged)
report_merge("TEST",  test_merged)

train_merged.to_csv(OUT_DIR / "windows_train_with_audio.csv", index=False)
val_merged.to_csv(OUT_DIR / "windows_val_with_audio.csv", index=False)
test_merged.to_csv(OUT_DIR / "windows_test_with_audio.csv", index=False)


In [ ]:
def get_audio_feature_columns(df: pd.DataFrame):
    drop = {
        "video_id","w_start","w_end","w_center","split",
        "mask_any","is_bg_negative",
        "y_MF","y_SK","y_SJ","mask_MF","mask_SK","mask_SJ",
        "has_any_audio_feature",
    }
    cols = [c for c in df.columns if c not in drop]
    cols = [c for c in cols if pd.api.types.is_numeric_dtype(df[c])]
    return cols

audio_feat_cols = get_audio_feature_columns(train_merged)
print("Audio feature columns:", len(audio_feat_cols))
audio_feat_cols[:20]


In [ ]:
import pandas as pd
import numpy as np

train = pd.read_csv(OUT_DIR / "windows_train_with_audio.csv")
val   = pd.read_csv(OUT_DIR / "windows_val_with_audio.csv")
test  = pd.read_csv(OUT_DIR / "windows_test_with_audio.csv")

for df in (train, val, test):
    if "w_center_x" in df.columns and "w_center_y" in df.columns:
        df.rename(columns={"w_center_x": "w_center"}, inplace=True)
        df.drop(columns=["w_center_y"], inplace=True)
    elif "w_center_x" in df.columns:
        df.rename(columns={"w_center_x": "w_center"}, inplace=True)
    elif "w_center_y" in df.columns:
        df.rename(columns={"w_center_y": "w_center"}, inplace=True)

DROP = {
    "video_id","w_start","w_end","w_center","split",
    "mask_any","is_bg_negative",
    "y_MF","y_SK","y_SJ","mask_MF","mask_SK","mask_SJ",
}
feat_cols = [c for c in train.columns if c not in DROP and pd.api.types.is_numeric_dtype(train[c])]

assert "has_any_audio_feature" in feat_cols

print("Audio feature columns:", len(feat_cols))
print("Missing-all-features (train):",
      train[[c for c in feat_cols if c != "has_any_audio_feature"]].isna().all(axis=1).mean())

mu = train[feat_cols].mean(skipna=True)
sd = train[feat_cols].std(skipna=True).replace(0, 1.0)

def transform(df):
    X = (df[feat_cols] - mu) / sd
    X = X.fillna(0.0)
    return X

X_train = transform(train)
X_val   = transform(val)
X_test  = transform(test)

print("X_train shape:", X_train.shape)
